# Poison-Resilient Continual NIDS — Project Results Storyboard

**Faculty presentation notebook · read-only evidence playback · no training**

**Research question.** Can a flow-based NIDS learn genuinely new attack classes while retaining earlier knowledge and limiting damage from corrupted replay labels?

This notebook tells the project story from dataset preparation through the custom tabular Transformer, static baseline, Task 1, replay, label poisoning, frozen-teacher filtering, quarantine/review, Task 2 evaluation, and saved diagnostic playback. The model is a **custom tabular Transformer trained from scratch on numeric flow features**, not a pretrained language Transformer.

**Evidence rule.** Missing artifacts are reported as unavailable, never converted into zero or invented results. Verified static/Task 1/historical Task 2 records may be used as clearly labelled fallbacks. The XGBoost diagnostic is shown **only when a matching attached artifact is selected**.

## 0. Configuration

This cell follows the same Kaggle pattern used by the experiment notebooks: **find the saved output folder under `/kaggle/input`, print the match, require exactly one match, then assign it**.

The prepared-data and Task 1 locations are already known. The attached `results_task2` dataset supplies the two ZIP files. The attached `full-train-v2` notebook supplies the saved `full_clean_20261003T045146_553311Z` folder. The attached `task2_XGboost` notebook supplies the saved `task2_xgboost_diagnostic` folder.

In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')

# 1) Existing verified inputs
PREPARED_DATA = Path('/kaggle/input/notebooks/dvdhere/preparation/review2_large_prepare_20261002T223303_224539Z')
TASK1_RUN_DIR = Path('/kaggle/input/notebooks/dvdhere/full-run-task1/full_task1_20261003T074354_763068Z')

assert PREPARED_DATA.is_dir(), f'Missing prepared folder: {PREPARED_DATA}'
assert TASK1_RUN_DIR.is_dir(), f'Missing Task 1 folder: {TASK1_RUN_DIR}'

# 2) Full clean static run from attached "full-train-v2" notebook output
full_clean_matches = [
    p for p in INPUT_ROOT.rglob('full_clean_20261003T045146_553311Z')
    if p.is_dir()
]
print('FULL CLEAN MATCHES:', *full_clean_matches, sep='\n')
assert len(full_clean_matches) == 1, 'Expected exactly one saved full_clean_20261003T045146_553311Z folder'
FULL_CLEAN_RUN = full_clean_matches[0]

# 3) Historical + retention/poisoning ZIPs from attached "results_task2" dataset
task2_dataset_roots = [
    p for p in INPUT_ROOT.iterdir()
    if p.is_dir() and p.name.lower().replace('-', '_') == 'results_task2'
]
print('RESULTS_TASK2 ROOTS:', *task2_dataset_roots, sep='\n')
assert len(task2_dataset_roots) == 1, 'Expected exactly one attached results_task2 dataset'
TASK2_DATASET_ROOT = task2_dataset_roots[0]

TASK2_HISTORICAL_ZIP = TASK2_DATASET_ROOT / 'results.zip'
TASK2_RETENTION_POISONING_ZIP = TASK2_DATASET_ROOT / 'results_ret.zip'

assert TASK2_HISTORICAL_ZIP.is_file(), f'Missing historical ZIP: {TASK2_HISTORICAL_ZIP}'
assert TASK2_RETENTION_POISONING_ZIP.is_file(), f'Missing retention/poisoning ZIP: {TASK2_RETENTION_POISONING_ZIP}'

# 4) XGBoost saved output from attached "task2_XGboost" notebook
xgb_matches = [
    p for p in INPUT_ROOT.rglob('task2_xgboost_diagnostic')
    if p.is_dir()
]
print('XGBOOST MATCHES:', *xgb_matches, sep='\n')
assert len(xgb_matches) == 1, 'Expected exactly one saved task2_xgboost_diagnostic folder'
XGBOOST_DIAGNOSTIC = xgb_matches[0]

# Not needed when the saved XGBoost output folder is attached.
XGBOOST_NOTEBOOK = None

REPO_ROOT = None

# Internal aliases used by the storyboard.
TASK1_RUN = TASK1_RUN_DIR
TASK2_HISTORICAL = TASK2_HISTORICAL_ZIP
TASK2_RETENTION_POISONING = TASK2_RETENTION_POISONING_ZIP

print('\nSELECTED INPUTS')
print('PREPARED_DATA:', PREPARED_DATA)
print('TASK1_RUN_DIR:', TASK1_RUN_DIR)
print('FULL_CLEAN_RUN:', FULL_CLEAN_RUN)
print('TASK2_HISTORICAL_ZIP:', TASK2_HISTORICAL_ZIP)
print('TASK2_RETENTION_POISONING_ZIP:', TASK2_RETENTION_POISONING_ZIP)
print('XGBOOST_DIAGNOSTIC:', XGBOOST_DIAGNOSTIC)

## 0.1 Safe discovery, readers, and plot output

The helper reads JSON/CSV/manifest evidence and named JSON members from ZIPs without extracting archives or loading checkpoints. Plots are written under a timestamped directory in `/kaggle/working` on Kaggle or `data/outputs` locally.

In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import json, zipfile, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from IPython.display import display

plt.rcParams.update({
    "figure.figsize": (10, 5.6), "figure.dpi": 120,
    "axes.titlesize": 15, "axes.labelsize": 11,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.fontsize": 9, "axes.grid": True, "grid.alpha": .22,
})
IS_KAGGLE = Path("/kaggle").exists()
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = ((Path("/kaggle/working") if IS_KAGGLE else Path("data/outputs"))
              / f"project_results_storyboard_{stamp}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def repo_root():
    if REPO_ROOT:
        p = Path(REPO_ROOT).expanduser()
        return p.resolve() if p.exists() else None
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q/"PROJECT_CONTEXT.md").exists() and (q/"README.md").exists():
            return q
    return None

REPO = repo_root()

def roots():
    r = []
    if REPO:
        r += [REPO/"data", REPO/"results", REPO]
    r += [Path("/kaggle/input"), Path("/kaggle/working")] if IS_KAGGLE else [Path.cwd()/"data", Path.cwd()/"results"]
    out = []
    for p in r:
        if p.exists():
            p = p.resolve()
            if p not in out: out.append(p)
    return out

def safe_zip_names(path):
    names = []
    with zipfile.ZipFile(path) as zf:
        for info in zf.infolist():
            pp = PurePosixPath(info.filename)
            mode = (info.external_attr >> 16) & 0o170000
            if pp.is_absolute() or ".." in pp.parts or info.flag_bits & 1 or mode == 0o120000:
                raise ValueError(f"Unsafe/encrypted/symlink ZIP member: {info.filename}")
            names.append(info.filename)
    return names

def zip_has(path, tokens):
    try:
        blob = "\n".join(safe_zip_names(path)).lower()
    except Exception:
        return False
    return all(t.lower() in blob for t in tokens)

def uniq(paths):
    out, seen = [], set()
    for p in paths:
        key = str(Path(p).resolve())
        if key not in seen:
            seen.add(key); out.append(Path(p))
    return out

def discover(kind):
    hits = []
    for root in roots():
        if kind == "prepared":
            for p in root.rglob("split_manifest.json"):
                d=p.parent
                if (d/"features_manifest.json").exists() and (d/"preprocessing_static.json").exists(): hits.append(d)
        elif kind == "full_clean":
            for p in root.rglob("test_metrics.json"):
                d=p.parent
                if "full_clean" in d.name.lower() and ((d/"history.json").exists() or (d/"training_manifest.json").exists()): hits.append(d)
            for z in root.rglob("*.zip"):
                if zip_has(z, ["full_clean_", "test_metrics.json", "training_manifest.json"]): hits.append(z)
        elif kind == "task1":
            for n in ("task1_test_metrics.json","old_test_metrics.json"):
                for p in root.rglob(n):
                    d=p.parent
                    if (d/"novelty.json").exists() or (d/"replay_buffer.npz").exists(): hits.append(d)
            for z in root.rglob("*.zip"):
                if zip_has(z, ["full_task1_", "novelty.json", "replay_buffer.npz"]): hits.append(z)
        elif kind == "task2_historical":
            for p in root.rglob("comparison.json"):
                if "task2" in str(p.parent).lower(): hits.append(p.parent)
            for z in root.rglob("*.zip"):
                if zip_has(z, ["full_task2_targeted_", "comparison.json"]): hits.append(z)
        elif kind == "task2_retention":
            for p in root.rglob("gate_audit.json"):
                s=str(p.parent).lower()
                if "retention" in s or "poison" in s or "task2_b" in s: hits.append(p.parent)
            for z in root.rglob("*.zip"):
                if zip_has(z, ["gate_audit.json"]) and (zip_has(z, ["retention"]) or zip_has(z, ["task2_b_poisoning"])): hits.append(z)
        elif kind == "xgboost":
            for p in root.rglob("validation_comparison.json"):
                d=p.parent
                if "xgboost" in str(d).lower() and list(d.rglob("test_metrics.json")): hits.append(d)
            for z in root.rglob("*.zip"):
                if zip_has(z, ["xgboost","test_metrics.json","validation_comparison.json"]): hits.append(z)
            for p in root.rglob("*.ipynb"):
                if "xgboost" in p.name.lower(): hits.append(p)
    return uniq(hits)

def select(label, explicit, kind):
    if explicit:
        p=Path(explicit).expanduser()
        if p.exists():
            print(f"{label}: explicit -> {p}"); return p
        print(f"{label}: configured path does not exist: {p}"); return None
    c=discover(kind)
    if len(c)==1:
        print(f"{label}: unique auto-discovery -> {c[0]}"); return c[0]
    if len(c)==0:
        print(f"{label}: not available in this checkout / attached inputs."); return None
    print(f"{label}: {len(c)} candidates found; set the selector explicitly:")
    for p in c: print(" -", p)
    return None

ARTIFACTS = {
    "prepared": select("Prepared data",PREPARED_DATA,"prepared"),
    "full_clean": select("Full clean run",FULL_CLEAN_RUN,"full_clean"),
    "task1": select("Task 1 run",TASK1_RUN,"task1"),
    "task2_historical": select("Historical Task 2",TASK2_HISTORICAL,"task2_historical"),
    "task2_retention": select("Retention/poisoning follow-up",TASK2_RETENTION_POISONING,"task2_retention"),
    "xgboost": select("XGBoost diagnostic", XGBOOST_DIAGNOSTIC if XGBOOST_DIAGNOSTIC is not None else XGBOOST_NOTEBOOK, "xgboost"),
}

def read_json(source, names):
    if source is None: return None, None
    source=Path(source); names=tuple(names)
    if source.is_dir():
        m=[]
        for n in names: m += list(source.rglob(n))
        m=uniq(m)
        if len(m)!=1: return None, f"{source} (expected one of {names}; found {len(m)})"
        return json.loads(m[0].read_text()), str(m[0])
    if source.suffix.lower()==".zip":
        zn=safe_zip_names(source)
        m=[n for n in zn if PurePosixPath(n).name in names]
        if len(m)!=1: return None, f"{source} (expected one of {names}; found {len(m)})"
        with zipfile.ZipFile(source) as zf:
            return json.loads(zf.read(m[0]).decode()), f"{source}!/{m[0]}"
    return None, str(source)

def notebook_output_text(path):
    """Collect saved text outputs from an executed .ipynb without executing it."""
    p=Path(path)
    nb=json.loads(p.read_text())
    chunks=[]
    for cell in nb.get("cells",[]):
        for out in cell.get("outputs",[]):
            if isinstance(out.get("text"),list):
                chunks.append("".join(out["text"]))
            elif isinstance(out.get("text"),str):
                chunks.append(out["text"])
            data=out.get("data",{})
            plain=data.get("text/plain") if isinstance(data,dict) else None
            if isinstance(plain,list): chunks.append("".join(plain))
            elif isinstance(plain,str): chunks.append(plain)
    return "\n".join(chunks)

def metrics(obj):
    if not isinstance(obj,dict): return None
    for k in ("metrics","test_metrics","combined_test_metrics"):
        if isinstance(obj.get(k),dict) and "accuracy" in obj[k]: return obj[k]
    return obj if "accuracy" in obj else None

def benign_fpr(m):
    if not isinstance(m,dict): return None
    for k in ("benign_fpr","benign_false_positive_rate"):
        if isinstance(m.get(k),(int,float)): return float(m[k])
    b=m.get("benign_false_positive")
    return float(b["rate"]) if isinstance(b,dict) and isinstance(b.get("rate"),(int,float)) else None

def save(fig,name,evidence):
    p=OUTPUT_DIR/name
    fig.tight_layout(); fig.savefig(p,bbox_inches="tight"); plt.show()
    print("Saved:",p); print("Evidence:",evidence)
    return p

print("Repository:", REPO or "not found; labelled verified fallbacks remain usable")
print("Plot output:", OUTPUT_DIR)

# 1. Project overview

**Pipeline:** raw flow data → preprocessing → custom tabular Transformer → static baseline / Task 1 → replay → label poisoning → frozen-teacher gate → quarantine/review → Task 2 evaluation / saved playback.

Quarantine here is a **training-data review action** for suspicious replay rows. It is not equivalent to the NIDS deciding that a live network flow is malicious.

In [ ]:
stages=["Raw flow\ndata","Train-only\npreprocessing","Custom tabular\nTransformer","Static baseline\n/ Task 1","Replay","Label\npoisoning","Frozen-teacher\ngate","Quarantine\n/ review","Task 2 evaluation\n/ playback"]
fig,ax=plt.subplots(figsize=(15,4.5)); ax.set_xlim(0,len(stages)); ax.set_ylim(0,1); ax.axis("off")
for i,label in enumerate(stages):
    x=i+.08
    ax.add_patch(FancyBboxPatch((x,.35),.84,.32,boxstyle="round,pad=.02",fill=False,linewidth=1.4))
    ax.text(x+.42,.51,label,ha="center",va="center",fontsize=9)
    if i<len(stages)-1:
        ax.annotate("",xy=(i+1.06,.51),xytext=(i+.93,.51),arrowprops=dict(arrowstyle="->",linewidth=1.4))
ax.set_title("Project pipeline — research evidence flow")
save(fig,"01_project_pipeline.png","ARCHITECTURE.md + observed experiment lineage; no training executed here.")

# 2. Dataset and preparation

The project uses the **Kaggle CIC-IDS-Collection parquet used by this project**, stored during development as `data/cic-collection.parquet`. The repository audit conservatively calls it a combined flow collection because there is no per-row source-release identifier. We do **not** claim that the original ~6 GB AWS CSE-CIC-IDS2018 release was downloaded.

The **8 broad `ClassLabel` categories** and **33 detailed `Label` values** are different label granularities over the same data. They are **not 41 classes**.

In [ ]:
CLASS_ORDER=["Benign","DDoS","DoS","Botnet","Bruteforce","Infiltration","Webattack","Portscan"]
CLASS_COUNTS=[7_186_189,1_234_729,397_344,145_968,103_244,94_857,2_995,2_255]
SPLITS={"Train":6_417_308,"Validation":1_375_139,"Test":1_375_134}
ledger=pd.DataFrame([
    ("Flow records",9_167_581),("Total columns",59),("Numeric candidate features",57),
    ("Retained features",54),("Broad ClassLabel classes",8),("Detailed Label values",33)
],columns=["Dataset ledger","Value"])
display(ledger.style.hide(axis="index").format({"Value":"{:,}"}))
print("Evidence: data/metadata/combined_flow_collection_audit.md")

cdf=pd.DataFrame({"Class":CLASS_ORDER,"Rows":CLASS_COUNTS}); display(cdf.style.hide(axis="index").format({"Rows":"{:,}"}))
fig,ax=plt.subplots(); ax.bar(cdf["Class"],cdf["Rows"]); ax.set_yscale("log"); ax.set_ylabel("Rows (log scale)")
ax.set_title("Broad-class counts — severe imbalance"); ax.tick_params(axis="x",rotation=30)
for i,v in enumerate(CLASS_COUNTS): ax.text(i,v*1.08,f"{v:,}",ha="center",fontsize=8)
save(fig,"02_class_counts.png","data/metadata/combined_flow_collection_audit.md")

sm,ss=read_json(ARTIFACTS["prepared"],["split_manifest.json"])
active=SPLITS
if isinstance(sm,dict):
    try:
        obs={"Train":sum(int(v["train"]) for v in sm["counts"].values()),
             "Validation":sum(int(v["validation"]) for v in sm["counts"].values()),
             "Test":sum(int(v["test"]) for v in sm["counts"].values())}
        if obs==SPLITS: active=obs; split_src=ss
        else: split_src="verified PROJECT_STATUS.md totals (attached manifest differed)"
    except Exception: split_src="verified PROJECT_STATUS.md totals"
else: split_src="verified PROJECT_STATUS.md full-row preparation totals"

sdf=pd.DataFrame({"Partition":list(active),"Rows":list(active.values())}); display(sdf.style.hide(axis="index").format({"Rows":"{:,}"}))
fig,ax=plt.subplots(figsize=(8,5)); ax.bar(sdf["Partition"],sdf["Rows"]); ax.set_ylabel("Rows"); ax.set_title("Fixed train / validation / test split")
for i,v in enumerate(sdf["Rows"]): ax.text(i,v*1.015,f"{v:,}",ha="center")
save(fig,"03_split_counts.png",split_src)
print("Static preprocessing fit scope: all authorized training rows.")
print("Continual preprocessing fit scope: Task 1 training IDs 0–4 only, then frozen.")

# 3. Model and learning setup

The full clean experiment used **54 features, hidden dimension 64, 4 attention heads, 2 encoder layers, feed-forward dimension 128, batch size 256, learning rate 0.001, seed 42**.

- **Hidden dimension:** width of each learned numeric feature-token representation.
- **Heads:** parallel attention patterns across feature tokens.
- **Replay:** old-class examples reintroduced while learning Task 2 to limit forgetting.
- **Frozen teacher:** unchanged Task 1 model used to judge replay-label consistency.
- **Label-flipping poison:** controlled corruption of training labels.
- **Quarantine:** review queue for gate-rejected training replay rows, not a firewall.

In [ ]:
VERIFIED_CFG={"num_features":54,"hidden_dim":64,"num_heads":4,"num_layers":2,"mlp_dim":128,"batch_size":256,"learning_rate":.001,"seed":42}
co,cs=read_json(ARTIFACTS["full_clean"],["training_manifest.json","run_config.json"])
active=VERIFIED_CFG.copy(); cfg_src="verified full-clean training_manifest.json / notebook"
if isinstance(co,dict):
    c=co.get("config") or co
    parsed={k:c.get(k) for k in active if k in c}
    if len(parsed)==len(active): active=parsed; cfg_src=cs
display(pd.DataFrame([
 ("Retained features",active["num_features"]),("Hidden dimension",active["hidden_dim"]),
 ("Attention heads",active["num_heads"]),("Encoder layers",active["num_layers"]),
 ("Feed-forward / MLP",active["mlp_dim"]),("Batch size",active["batch_size"]),
 ("Learning rate",active["learning_rate"]),("Seed",active["seed"])
],columns=["Configuration","Value"]).style.hide(axis="index"))
print("Evidence:",cfg_src)

# 4. Clean static eight-class baseline

The saved full-data test result is intentionally shown with per-class metrics. High aggregate accuracy does not erase weak rare-class recall. Training curves render only when the full run with `history.json` is attached.

In [ ]:
STATIC={
 "accuracy":.9828620338090688,"macro_f1":.757755762638612,"balanced_accuracy":.7291289708762002,"benign_fpr":.00445669840657261,
 "per_class":{
 "0":{"name":"Benign","precision":.9839054239656434,"recall":.9955433015934274,"f1":.9896901511846781,"support":1077928},
 "1":{"name":"DDoS","precision":.9923284274084856,"recall":.9938339929485068,"f1":.9930806395485285,"support":185209},
 "2":{"name":"DoS","precision":.9394637935354547,"recall":.9436251069612926,"f1":.941539852342926,"support":59601},
 "3":{"name":"Botnet","precision":.9985207784403458,"recall":.9865722767755195,"f1":.9925105679103107,"support":21895},
 "4":{"name":"Bruteforce","precision":.9984427718660783,"recall":.9936717034741056,"f1":.9960515243705093,"support":15486},
 "5":{"name":"Infiltration","precision":.4858463316002311,"recall":.059108799550182736,"f1":.10539507487937841,"support":14228},
 "6":{"name":"Webattack","precision":1.0,"recall":.12694877505567928,"f1":.22529644268774704,"support":449},
 "7":{"name":"Portscan","precision":.9253731343283582,"recall":.7337278106508875,"f1":.8184818481848185,"support":338}},
 "confusion_matrix":[[1073124,553,3320,17,11,890,0,13],[878,184067,264,0,0,0,0,0],[2552,794,56241,0,13,0,0,1],
 [283,7,4,21601,0,0,0,0],[90,2,2,0,15388,0,0,4],[13328,22,27,8,0,841,0,2],[384,0,1,7,0,0,57,0],[39,45,6,0,0,0,0,248]]
}
to,ts=read_json(ARTIFACTS["full_clean"],["test_metrics.json"]); am=metrics(to)
m=am if isinstance(am,dict) and "per_class" in am and "confusion_matrix" in am else STATIC
src=ts if m is am else "verified fallback: full_clean_20261003T045146_553311Z/test_metrics.json"
display(pd.DataFrame([
 ("Test accuracy",100*m["accuracy"]),("Macro-F1",100*m["macro_f1"]),
 ("Balanced accuracy",100*m.get("balanced_accuracy",np.nan)),
 ("Benign FPR",100*(benign_fpr(m) if benign_fpr(m) is not None else STATIC["benign_fpr"]))
],columns=["Metric","Percent"]).style.hide(axis="index").format({"Percent":"{:.4f}%"}))
rows=[{"Class":r["name"],"Precision":r["precision"],"Recall":r["recall"],"F1":r["f1"],"Support":r["support"]} for r in m["per_class"].values()]
pdf=pd.DataFrame(rows); display(pdf.style.hide(axis="index").format({"Precision":"{:.2%}","Recall":"{:.2%}","F1":"{:.2%}","Support":"{:,}"}))
fig,ax=plt.subplots(); x=np.arange(8); w=.36
ax.bar(x-w/2,pdf["Recall"]*100,width=w,label="Recall"); ax.bar(x+w/2,pdf["F1"]*100,width=w,label="F1")
ax.set_xticks(x,pdf["Class"],rotation=30,ha="right"); ax.set_ylim(0,105); ax.set_ylabel("Percent"); ax.set_title("Static per-class recall and F1"); ax.legend()
save(fig,"04_static_per_class.png",src)
cm=np.asarray(m["confusion_matrix"],float); rn=np.divide(cm,cm.sum(1,keepdims=True),out=np.zeros_like(cm),where=cm.sum(1,keepdims=True)!=0)
fig,ax=plt.subplots(figsize=(8.5,7)); im=ax.imshow(rn,vmin=0,vmax=1,aspect="auto")
ax.set_xticks(range(8),CLASS_ORDER,rotation=35,ha="right"); ax.set_yticks(range(8),CLASS_ORDER); ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Row-normalized static confusion matrix")
for i in range(8):
    for j in range(8): ax.text(j,i,f"{100*rn[i,j]:.1f}",ha="center",va="center",fontsize=7)
fig.colorbar(im,ax=ax,label="Row fraction"); save(fig,"05_static_confusion.png",src)
ho,hs=read_json(ARTIFACTS["full_clean"],["history.json"]); ep=ho.get("epochs") if isinstance(ho,dict) else None
if ep:
    h=pd.DataFrame(ep)
    fig,ax=plt.subplots(); ax.plot(h["epoch"],h["train_loss"],marker="o",label="Train loss"); ax.plot(h["epoch"],h["validation_loss"],marker="o",label="Validation loss")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("Full clean loss history"); ax.legend(); save(fig,"06_training_loss.png",hs)
    fig,ax=plt.subplots(); ax.plot(h["epoch"],h["validation_accuracy"]*100,marker="o"); ax.set_xlabel("Epoch"); ax.set_ylabel("Validation accuracy (%)"); ax.set_title("Full clean validation accuracy"); save(fig,"07_validation_accuracy.png",hs)
else:
    print("Training curves: not available in this checkout. Set FULL_CLEAN_RUN to a complete run containing history.json.")
print(f"Rare-class recall — Infiltration {100*m['per_class']['5']['recall']:.2f}%, Webattack {100*m['per_class']['6']['recall']:.2f}%, Portscan {100*m['per_class']['7']['recall']:.2f}%.")

# 5. Continual-learning journey

**Task 1 old IDs 0–4:** Benign, DDoS, DoS, Botnet, Bruteforce.  
**Task 2 new IDs 5–7:** Infiltration, Webattack, Portscan.

Replay counters catastrophic forgetting by preserving exposure to old training examples while Task 2 introduces the new classes. The continual preprocessor is fitted only on Task 1 training rows and then frozen; this is distinct from the static eight-class preprocessing fit.

In [ ]:
display(pd.DataFrame([(i,c,"Task 1 / old" if i<5 else "Task 2 / new") for i,c in enumerate(CLASS_ORDER)],columns=["ID","Class","Role"]).style.hide(axis="index"))
fig,ax=plt.subplots(figsize=(11,3.5)); ax.set_xlim(0,10); ax.set_ylim(0,1); ax.axis("off")
events=[(1.2,"Task 1\nIDs 0–4"),(3.5,"Freeze teacher\n+ preprocessor"),(5.6,"Select old\nreplay"),(7.6,"Task 2\nIDs 5–7"),(9,"Evaluate old/new\n+ forgetting")]
ax.annotate("",xy=(9.5,.5),xytext=(.5,.5),arrowprops=dict(arrowstyle="->",linewidth=2))
for n,(x0,t) in enumerate(events):
    ax.plot([x0,x0],[.42,.58],linewidth=2); ax.text(x0,.70 if n%2==0 else .27,t,ha="center",va="center")
ax.set_title("Continual-learning timeline"); save(fig,"08_continual_timeline.png","ARCHITECTURE.md + verified Task 1 evidence")
TASK1={"accuracy":.9925734439413022,"macro_f1":.981162951652861,"balanced_accuracy":.9784715590433037,"benign_fpr":.004171892742372403,
       "novelty_unknown_recall":.04602064602064602,"novelty_known_false_reject":.04998606739557348}
o,s=read_json(ARTIFACTS["task1"],["task1_test_metrics.json","old_test_metrics.json"]); mm=metrics(o)
if isinstance(mm,dict):
    t={"accuracy":mm["accuracy"],"macro_f1":mm["macro_f1"],"balanced_accuracy":mm.get("balanced_accuracy",np.nan),"benign_fpr":benign_fpr(mm)}
    task_src=s
else:
    t=TASK1; task_src="verified fallback: full_task1_20261003T074354_763068Z"
display(pd.DataFrame([("Old-class accuracy",100*t["accuracy"]),("Old-class macro-F1",100*t["macro_f1"]),("Balanced accuracy",100*t["balanced_accuracy"]),("Benign FPR",100*t["benign_fpr"])],columns=["Task 1 metric","Percent"]).style.hide(axis="index").format({"Percent":"{:.3f}%"}))
print("Evidence:",task_src)
display(pd.DataFrame([("Static","all eight training classes","static eight-class baseline"),("Continual","Task 1 training IDs 0–4 only","Task 1 teacher + Task 2")],columns=["Preprocessor","Fit scope","Use"]).style.hide(axis="index"))
print(f"Known Task 1 novelty limitation: unknown recall {100*TASK1['novelty_unknown_recall']:.2f}% at ~{100*TASK1['novelty_known_false_reject']:.2f}% known false rejection.")

# 6. Poisoning and quarantine experiment

Historical full Task 2 compares **clean replay / targeted poisoned replay / targeted poisoned replay + frozen-teacher filter**. The gate's rejection is a decision about whether a suspicious **training replay row** should enter filtered training; it is not a malicious-flow detection result.

In [ ]:
T2={
 "clean":{"accuracy":.6010047020872148,"macro_f1":.5664584687018178,"benign_fpr":.4989118011592611,"dos_recall":.9715273233670576,"old_to_benign":62},
 "poisoned":{"accuracy":.44913586603196487,"macro_f1":.5379282266745713,"benign_fpr":.6885997951625712,"dos_recall":.9605879095988322,"old_to_benign":3641},
 "filtered":{"accuracy":.6077218656509111,"macro_f1":.5578505675817005,"benign_fpr":.4666276411782605,"dos_recall":.6072884683142901,"old_to_benign":6856}}
src="docs/evidence/task2_targeted_summary.json (verified compact record of data/task2/results.zip)"
keys=["clean","poisoned","filtered"]; names=["Clean","Poisoned","Filtered"]
d=pd.DataFrame({"Arm":names,"Accuracy":[T2[k]["accuracy"] for k in keys],"Macro-F1":[T2[k]["macro_f1"] for k in keys],"Benign FPR":[T2[k]["benign_fpr"] for k in keys],"DoS recall":[T2[k]["dos_recall"] for k in keys]})
display(d.style.hide(axis="index").format({"Accuracy":"{:.2%}","Macro-F1":"{:.2%}","Benign FPR":"{:.2%}","DoS recall":"{:.2%}"}))
fig,ax=plt.subplots(); x=np.arange(3); w=.25
ax.bar(x-w,d["Accuracy"]*100,width=w,label="Accuracy"); ax.bar(x,d["Macro-F1"]*100,width=w,label="Macro-F1"); ax.bar(x+w,d["Benign FPR"]*100,width=w,label="Benign FPR")
ax.set_xticks(x,names); ax.set_ylabel("Percent"); ax.set_title("Task 2 poisoning / mitigation trade-off"); ax.legend(); save(fig,"09_task2_three_arm.png",src)
g=pd.DataFrame([("Poison rejected",80,80,1.),("Clean rejected",42,420,42/420),("Rows retained",378,500,378/500)],columns=["Gate outcome","Rows","Denominator","Rate"])
display(g.style.hide(axis="index").format({"Rate":"{:.2%}"}))
fig,ax=plt.subplots(); vals=[80,42,378]; labels=["Poison rejected","Clean rejected","Retained"]; ax.bar(labels,vals); ax.set_ylabel("Rows"); ax.set_title("Frozen-teacher gate audit — 500 replay candidates")
for i,v in enumerate(vals): ax.text(i,v+6,str(v),ha="center")
save(fig,"10_gate_outcomes.png",src)
print("Conclusion: poisoning degraded the study. Filtering improved accuracy relative to poisoning and lowered Benign FPR, but DoS recall fell to 60.73% and old-attack→Benign errors rose to 6,856. This single-seed result is not proof of universal robustness.")
print("Gate rejection != NIDS attack detection.")
if ARTIFACTS["task2_historical"] is None: print("Original historical Task 2 artifact not attached; verified compact evidence is used.")
if ARTIFACTS["task2_retention"] is None: print("Later retention/poisoning artifact: not available in this checkout.")

# 7. XGBoost diagnostic — verified results artifact or executed notebook

The XGBoost run is a **joint-training diagnostic baseline**, not the continual poison-resilient model, not a poisoning defense, and not a final deployed model.

You may attach either the diagnostic results ZIP/folder **or the executed XGBoost Kaggle notebook**. When an `.ipynb` is selected, this storyboard reads its saved text outputs only; it does not execute any archived code. The result is rendered only when the reviewed metric fingerprint is present.

In [ ]:
EXPECTED={"accuracy":.8200640810277399,"macro_f1":.7595821321323564,"benign_fpr":.22734078713977185,"inf_precision":.05040836863167256,"inf_recall":.9083497329210009}

# Verified per-class test precision/recall from the reviewed XGBoost diagnostic.
# These rows are displayed only after an attached artifact/notebook matches EXPECTED.
XGB_PER_CLASS=[
 ("Benign",.99775,.77266),("DDoS",.99935,.99617),("DoS",.99249,.99520),
 ("Botnet",.98847,.99804),("Bruteforce",.99890,.99871),
 ("Infiltration",.050408,.908350),("Webattack",.33638,.98218),("Portscan",.46078,.97337)
]

def pc(m,i):
    p=m.get("per_class",{}) if isinstance(m,dict) else {}
    return p.get(str(i)) or p.get(i)

def notebook_has_xgb_fingerprint(path):
    text=notebook_output_text(path)
    nums=[]
    for token in re.findall(r"(?<![A-Za-z_])-?\d+(?:\.\d+)?",text):
        try: nums.append(float(token))
        except ValueError: pass
    def seen(target):
        return any(abs(v-target) <= 5e-5 or abs(v-target*100) <= 5e-3 for v in nums)
    return all(seen(v) for v in EXPECTED.values()), text

xsrc=ARTIFACTS["xgboost"]
xm=None; xs=None; ok=False

if xsrc is not None and Path(xsrc).suffix.lower()==".ipynb":
    try:
        ok,_=notebook_has_xgb_fingerprint(xsrc)
        xs=str(xsrc)+" (saved executed notebook outputs)"
        if ok:
            xm={
              "accuracy":EXPECTED["accuracy"],
              "macro_f1":EXPECTED["macro_f1"],
              "benign_fpr":EXPECTED["benign_fpr"],
              "per_class":{
                str(i):{"name":name,"precision":prec,"recall":rec}
                for i,(name,prec,rec) in enumerate(XGB_PER_CLASS)
              }
            }
    except Exception as exc:
        print("Could not inspect XGBoost notebook:",exc)
else:
    xo,xs=read_json(xsrc,["test_metrics.json","selected_test_metrics.json"])
    xm=metrics(xo)
    if isinstance(xm,dict):
        try:
            inf=pc(xm,5)
            checks=[
              math.isclose(float(xm["accuracy"]),EXPECTED["accuracy"],abs_tol=5e-6),
              math.isclose(float(xm["macro_f1"]),EXPECTED["macro_f1"],abs_tol=5e-6),
              math.isclose(float(benign_fpr(xm)),EXPECTED["benign_fpr"],abs_tol=5e-6),
              math.isclose(float(inf["precision"]),EXPECTED["inf_precision"],abs_tol=5e-6),
              math.isclose(float(inf["recall"]),EXPECTED["inf_recall"],abs_tol=5e-6)]
            ok=all(checks)
        except Exception:
            ok=False

if not ok:
    print("XGBoost diagnostic: not available, ambiguous, or the selected artifact/notebook does not match the reviewed fingerprint.")
    print("Set XGBOOST_NOTEBOOK to the executed Kaggle .ipynb, or XGBOOST_DIAGNOSTIC to the verified results ZIP/folder.")
else:
    inf=pc(xm,5)
    display(pd.DataFrame([
      ("Accuracy",100*xm["accuracy"]),("Macro-F1",100*xm["macro_f1"]),
      ("Infiltration recall",100*inf["recall"]),("Infiltration precision",100*inf["precision"]),
      ("Benign FPR",100*benign_fpr(xm))
    ],columns=["Diagnostic metric","Percent"]).style.hide(axis="index").format({"Percent":"{:.4f}%"}))
    rows=[]
    for i in range(8):
        r=pc(xm,i)
        if r: rows.append((r.get("name",CLASS_ORDER[i]),100*r["precision"],100*r["recall"]))
    xd=pd.DataFrame(rows,columns=["Class","Precision","Recall"])
    display(xd.style.hide(axis="index").format({"Precision":"{:.2f}%","Recall":"{:.2f}%"}))
    fig,ax=plt.subplots(); x=np.arange(len(xd)); w=.36
    ax.bar(x-w/2,xd["Precision"],width=w,label="Precision"); ax.bar(x+w/2,xd["Recall"],width=w,label="Recall")
    ax.set_xticks(x,xd["Class"],rotation=30,ha="right"); ax.set_ylim(0,105); ax.set_ylabel("Percent")
    ax.set_title("XGBoost diagnostic — precision vs recall"); ax.legend(); save(fig,"11_xgboost_precision_recall.png",xs)
    print("Diagnostic label: joint training only — no continual-learning or poisoning-defense claim.")

# 8. Honest final dashboard

The defensible conclusion is not “we solved NIDS.” We built a reproducible research pipeline, demonstrated concrete continual-learning and poisoning mechanisms, measured where the filter helped, and recorded where the system still fails.

In [ ]:
dem=pd.DataFrame([
 ("Full-row preparation","9,167,581 rows, fixed splits, 54 retained features, train-only fitting."),
 ("Custom tabular Transformer","From-scratch feature-token Transformer; full static run completed."),
 ("Static baseline","98.2862% overall accuracy, but rare-class recall remained weak."),
 ("Task 1","Five old classes, separate continual preprocessing, saved replay/teacher evidence."),
 ("Replay / Task 2","Measured forgetting versus acquisition trade-offs."),
 ("Label poisoning","Deterministic training-label corruption, not packet manipulation."),
 ("Frozen-teacher gate","Rejected suspicious replay labels and audited poison/clean rejects."),
 ("Quarantine","Training-data review semantics kept separate from flow classification.")
],columns=["What we demonstrated","Evidence-backed meaning"])
display(dem.style.hide(axis="index"))
lim=pd.DataFrame([
 ("Rare-class imbalance","Infiltration/Webattack weaknesses make accuracy alone misleading."),
 ("Single seed","Repeated-seed evaluation and uncertainty are still needed."),
 ("No independent second dataset","Cross-dataset generalization is untested."),
 ("No production firewall enforcement","Prototype classification/review is not policy enforcement."),
 ("Replay-gate false rejects","Clean replay rows can be rejected; downstream costs matter."),
 ("Missing/stale private artifacts","Unavailable evidence is labelled unavailable, never assumed."),
 ("Continual acquisition trade-off","Retention and new-class detection remain jointly difficult."),
 ("Repeated-seed evaluation","Required before stronger poison-resilience claims.")
],columns=["Current limitation / next step","Why it matters"])
display(lim.style.hide(axis="index"))
display(pd.DataFrame([(k,str(v) if v is not None else "not available / explicit selection required") for k,v in ARTIFACTS.items()],columns=["Optional evidence input","Resolved path"]).style.hide(axis="index"))
print("Figures:",OUTPUT_DIR)

## 8.1 30-second faculty explanation

> We built a flow-based NIDS research prototype around a custom tabular Transformer. We prepared 9.17 million flow records into fixed train, validation, and test partitions, trained a full eight-class static baseline, then separated five old classes from three new classes for continual learning. Replay helps preserve old knowledge, while controlled label flipping lets us study poisoning. A frozen Task 1 teacher can reject suspicious replay labels before Task 2 training, but the gate also rejects some clean rows and can hurt class-specific performance. Our strongest claim is a reproducible pipeline with measured attack/defense trade-offs—not universal poison resilience or production-ready eight-class detection.

### Five likely viva questions

**1. Are 8 ClassLabel classes plus 33 Label values equal to 41 classes?**  
No. They are broad and fine label granularities over the same rows.

**2. Why a Transformer?**  
Numeric features become learned tokens and self-attention models interactions among them. It is trained from scratch, not pretrained.

**3. Why replay?**  
Task 2 can cause catastrophic forgetting of Task 1; replay restores exposure to selected old examples.

**4. Does quarantine detect malicious network traffic?**  
No. The gate checks training replay-label consistency; rejected rows enter review. Flow classification is separate.

**5. Is the model proven poison-resilient?**  
No. The single-seed filtered arm improved some aggregate measures but hurt DoS recall and caused other errors. Repeated seeds, stronger attacks, and an independent dataset are needed.

## 9. Presentation readiness and exact inputs

The configuration now uses the same saved-output discovery pattern as the project's Kaggle experiment notebooks. It expects the four inputs visible in Kaggle: `preparation`, `full-train-v2`, `full-run-task1`, and `task2_XGboost`, plus the `results_task2` dataset containing `results.zip` and `results_ret.zip`.

In [ ]:
req=pd.DataFrame([
 ("PREPARED_DATA","Verified preparation notebook output folder"),
 ("TASK1_RUN_DIR","Verified full-run-task1 notebook output folder"),
 ("FULL_CLEAN_RUN","Auto-found full_clean_20261003T045146_553311Z under attached full-train-v2 output"),
 ("TASK2_HISTORICAL_ZIP","results_task2/results.zip"),
 ("TASK2_RETENTION_POISONING_ZIP","results_task2/results_ret.zip"),
 ("XGBOOST_DIAGNOSTIC","Auto-found task2_xgboost_diagnostic under attached task2_XGboost output")
],columns=["Selector","Resolved from Kaggle input"])
display(req.style.hide(axis="index"))
missing=[k for k,v in ARTIFACTS.items() if v is None]
print("Unavailable/ambiguous optional artifacts:",", ".join(missing) if missing else "none")
print("Fallback-ready: overview, dataset, model configuration, static test metrics/confusion, Task 1 summary, historical Task 2 poison/gate, final dashboard.")
print("Artifact-dependent: full-clean learning curves; direct attached-artifact playback; XGBoost diagnostic plot.")
print("Generated plots:",OUTPUT_DIR)